In [0]:
import cuml

In [0]:
%load_ext cuml.accel

In [0]:

import time
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
pio.renderers.default = "browser"

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from multiprocessing import Manager
from cuml.manifold import UMAP



In [0]:
path = "/Volumes/kumc_sleep/sleep_studies/shhs_data/test/"
X = np.load(path + "test_1.npy")

In [0]:
from cuml.cluster import KMeans

In [0]:
%pip list

In [0]:
    import cupy as cp
Z = cp.asarray(X, dtype=cp.float32)  # GPU array


In [0]:
!nvidia-smi    # shows driver + CUDA runtime

In [0]:
# ------------------------------------------------------------

# 1) Normalize RAW 512-D (angular geometry)

# ------------------------------------------------------------

X = normalize(Z.astype(np.float32), norm="l2")

N, D = X.shape

print("Clustering on RAW latent space:", X.shape)

In [0]:
# ------------------------------------------------------------

# 2) RANDOM HYPERPLANE LSH

# ------------------------------------------------------------

def lsh_hash(X, n_bits, seed=42):

    rng = np.random.default_rng(seed)

    hyperplanes = rng.standard_normal((n_bits, D)).astype(np.float32)

    proj = X @ hyperplanes.T

    bits = (proj > 0).astype(np.uint8)

    powers = (1 << np.arange(n_bits, dtype=np.uint64))

    return (bits.astype(np.uint64) * powers).sum(axis=1)
 
def bucket_stats(hash_vals):

    unique, counts = np.unique(hash_vals, return_counts=True)

    return {

        "nonempty": len(unique),

        "singleton_frac": float(np.sum(counts == 1)) / N,

        "median_size": float(np.median(counts)),

        "max_size": int(np.max(counts)),

    }
 
# ------------------------------------------------------------

# 3) FIND GOOD LSH BIT DEPTH

# ------------------------------------------------------------

bits_range = range(4, 22)

target_buckets = 2000
 
best_bits = None

best_score = None
 
for b in bits_range:

    h = lsh_hash(X, n_bits=b)

    stats = bucket_stats(h)

    score = (

        abs(stats["nonempty"] - target_buckets)

        + 3000 * stats["singleton_frac"]

    )

    print(f"bits={b} | buckets={stats['nonempty']} | singleton_frac={stats['singleton_frac']:.3f}")

    if best_score is None or score < best_score:

        best_score = score

        best_bits = b
 
print("Chosen LSH bits:", best_bits)
 
# ------------------------------------------------------------


In [0]:
# 4) LSH-MEANS INITIALIZATION

# ------------------------------------------------------------

def lsh_means_init(X, n_clusters, n_bits):

    h = lsh_hash(X, n_bits=n_bits)

    unique, inv, counts = np.unique(h, return_inverse=True, return_counts=True)
 
    # Largest buckets first

    order = np.argsort(counts)[::-1]

    keep = order[:5000]
 
    centroids = []

    weights = []
 
    for idx in keep:

        mask = inv == idx

        if mask.sum() == 0:

            continue

        c = X[mask].mean(axis=0)

        centroids.append(c)

        weights.append(mask.sum())
 
    C = np.vstack(centroids)

    weights = np.array(weights)
 
    km = MiniBatchKMeans(

        n_clusters=n_clusters,

        batch_size=4096,

        n_init=10,

        random_state=42

    )
 
    km.fit(C, sample_weight=weights)
 
    return normalize(km.cluster_centers_, norm="l2")


# ------------------------------------------------------------

# 5) SWITCH RATE

# ------------------------------------------------------------

def switch_rate(labels, night_id, time_idx):

    rates = []

    for nid in np.unique(night_id):

        m = night_id == nid

        if m.sum() < 2:

            continue

        t = time_idx[m]

        y = labels[m]

        order = np.argsort(t)

        y = y[order]

        rates.append(np.mean(y[1:] != y[:-1]))

    return float(np.mean(rates))
 
# ------------------------------------------------------------


In [0]:
# 6) SWEEP K ON RAW
# ------------------------------------------------------------

k_values = list(range(8, 18))
results = []
 
rng = np.random.default_rng(42)
sample_n = min(20000, len(X))
sil_idx = rng.choice(len(X), sample_n, replace = False)

X_sil = X[sil_idx] 

for k in k_values:

    init_centers = lsh_means_init(X, k, best_bits)

    km = MiniBatchKMeans(
        n_clusters=k,
        init=init_centers,
        n_init=1,
        batch_size=8192,
        max_iter=30,
        random_state=42
    )

    labels = km.fit_predict(X)
 
    sil = silhouette_score(X_sil, labels[sil_idx])
 
    #sw = switch_rate(labels, night_id, time_idx)
    
    sw = 0
    combined = sil
 
    results.append((k, sil, combined))

    print(f"k={k} | sil={sil:.4f} | combined={combined:.4f}")
 
    best_k = sorted(results, key=lambda x: x[3], reverse=True)[0][0]
    results.append((k, sil, sw, combined))

    print(f"k={k} | sil={sil:.4f} | switch={sw:.4f} | combined={combined:.4f}")
 
best_k = sorted(results, key=lambda x: x[3], reverse=True)[0][0]

print("Best k:", best_k)
 
# ------------------------------------------------------------

In [0]:
# 7) FINAL FIT

# ------------------------------------------------------------

final_init = lsh_means_init(X, best_k, best_bits)
 
final_km = MiniBatchKMeans(

    n_clusters=best_k,

    init=final_init,

    n_init=1,

    batch_size=8192,

    max_iter=300,

    random_state=42

)
 
cluster_id = final_km.fit_predict(X).astype(int)

print("Final cluster ids:", np.unique(cluster_id))
 
# ------------------------------------------------------------


In [0]:
# 8) UMAP ON RAW 512-D (NO PCA)

# ------------------------------------------------------------

reducer = umap.UMAP(

    n_neighbors=15,

    min_dist=0.1,

    n_components=3,

    metric="euclidean",

    random_state=42,

    verbose=True

)
 
embedding = reducer.fit_transform(X)
 
fig = go.Figure(

    data=go.Scattergl(

        x=embedding[:,0],

        y=embedding[:,1],

        z=embedding[:,2],

        mode="markers",

        marker=dict(

            size=3,

            color=cluster_id,

            showscale=True

        )

    )

)
 
fig.update_layout(

    title=f"RAW 512-D UMAP | k={best_k} | bits={best_bits}",

    height=800

)
 
fig.show()

 

In [0]:
# ------------------------------------------------------------
# Map cluster IDs to source zarr files
# ------------------------------------------------------------
import pandas as pd
from pathlib import Path
import numpy as np

# Load the cached metadata from Representation_Extracting
cache_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/cache_dir/")

# Find the most recent cache files
# Pattern: {variable}__{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}.npy
cache_files = list(cache_dir.glob("zarr_file_idx__PFTSleep*.npy"))

if len(cache_files) > 0:
    # Use the most recent cache
    cache_file = sorted(cache_files, key=lambda p: p.stat().st_mtime)[-1]
    cache_tag = cache_file.stem.replace("zarr_file_idx__", "")
    
    # Load all metadata arrays from Representation_Extracting
    zarr_file_idx = np.load(cache_dir / f"zarr_file_idx__{cache_tag}.npy")
    night_id = np.load(cache_dir / f"night_id__{cache_tag}.npy")
    time_idx = np.load(cache_dir / f"time_idx__{cache_tag}.npy")
    
    # Load the zarr files list
    zarr_files_list_path = cache_dir / f"zarr_files_list__{cache_tag}.npy"
    if zarr_files_list_path.exists():
        zarr_files_list = np.load(zarr_files_list_path, allow_pickle=True).tolist()
    else:
        # Fallback: reconstruct from zarr directory
        zarr_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")
        zarr_files_list = sorted([str(p) for p in zarr_dir.glob("*.zarr")])
    
    n_points = len(cluster_id)
    
    # Create a DataFrame with cluster assignments and full metadata
    results_df = pd.DataFrame({
        'point_idx': np.arange(n_points),
        'cluster_id': cluster_id,
        'zarr_file_idx': zarr_file_idx,
        'zarr_file_path': [zarr_files_list[idx] if idx < len(zarr_files_list) else 'unknown' for idx in zarr_file_idx],
        'zarr_file_name': [Path(zarr_files_list[idx]).name if idx < len(zarr_files_list) else 'unknown' for idx in zarr_file_idx],
        'night_id': night_id,
        'time_idx_sec': time_idx,
    })
    
    print(f"Loaded metadata from cache: {cache_tag}")
    print(f"Created mapping for {n_points} data points from {len(np.unique(zarr_file_idx))} zarr files")
    print(f"\nCluster distribution:")
    print(results_df['cluster_id'].value_counts().sort_index())
    print(f"\nData points per zarr file:")
    print(results_df.groupby('zarr_file_name').size().head(10))
    print(f"\nFirst few rows:")
    display(results_df.head(10))
    
else:
    print("No cache files found. Please run Representation_Extracting first.")
    print("Expected cache location:", cache_dir)

In [0]:
# ------------------------------------------------------------
# Function to handle multiple zarr files
# (Use this when processing multiple files from Representation_Extracting)
# ------------------------------------------------------------

def create_multi_file_mapping(latent_dir, cluster_ids):
    """
    Create mapping for data points from multiple zarr files.
    
    Parameters:
    -----------
    latent_dir : Path
        Directory containing .npy files with latent representations
    cluster_ids : np.ndarray
        Cluster assignments for all points
    
    Returns:
    --------
    pd.DataFrame with columns: point_idx, cluster_id, source_file, 
                               file_idx, window_idx_in_file
    """
    latent_path = Path(latent_dir)
    npy_files = sorted(latent_path.glob("*.npy"))
    
    records = []
    global_idx = 0
    
    for file_idx, npy_file in enumerate(npy_files):
        # Load to get shape
        data = np.load(npy_file)
        n_windows = len(data)
        
        # Create records for this file
        for local_idx in range(n_windows):
            records.append({
                'point_idx': global_idx,
                'cluster_id': cluster_ids[global_idx],
                'source_file': npy_file.name,
                'file_idx': file_idx,
                'window_idx_in_file': local_idx
            })
            global_idx += 1
    
    return pd.DataFrame(records)

# Example usage (commented out for now):
latent_dir = "/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/cache_dir/"
multi_file_df = create_multi_file_mapping(latent_dir, cluster_id)
multi_file_df.to_csv(output_dir / "multi_file_cluster_assignments.csv", index=False)

print("Multi-file mapping function defined.")
print("Use this when processing multiple zarr files from Representation_Extracting.")